In [1]:
import numpy as np
import time
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

print("=================================================================")
print("STARTING CAPSTONE: PRODUCTION RAG PIPELINE & QA EVALUATION SUITE")
print("=================================================================\n")

# STEP 1: PINECONE CLOUD INITIALIZATION & CONNECTIVITY
PINECONE_API_KEY = "<Enter your APIs>"  # <-- Paste your token here
pc = Pinecone(api_key=PINECONE_API_KEY)

capstone_index_name = "capstone-qa-index"
DIMENSIONS = 384  # Matches our Sentence-Transformer model geometry

# Safely provision a clean capstone index over our AWS region subnet
if capstone_index_name not in pc.list_indexes().names():
    print(f"[INFRA] Creating cloud serverless index: '{capstone_index_name}'...")
    pc.create_index(
        name=capstone_index_name,
        dimension=DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    while not pc.describe_index(capstone_index_name).status['ready']:
        time.sleep(1)

index = pc.Index(capstone_index_name)
print(f"[INFRA] Successfully connected to Pinecone cloud cluster.")

STARTING CAPSTONE: PRODUCTION RAG PIPELINE & QA EVALUATION SUITE

[INFRA] Creating cloud serverless index: 'capstone-qa-index'...
[INFRA] Successfully connected to Pinecone cloud cluster.


In [2]:
# STEP 2: LOAD LOCAL CPU EMBEDDING MODEL
print("[MODEL] Loading CPU-optimized 'all-MiniLM-L6-v2' framework...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

[MODEL] Loading CPU-optimized 'all-MiniLM-L6-v2' framework...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# STEP 3: DATASET INGESTION (SQuAD Corporate Infrastructure QA Format)
qa_knowledge_corpus = [
    {"id": "doc_001", "text": "Our current operational AI infrastructure cluster runs on a shared AWS t3.2xlarge instance costing $11.65 per month."},
    {"id": "doc_002", "text": "Persistent storage architecture is allocated at 200 GB General Purpose EBS costing $16.00 monthly."},
    {"id": "doc_003", "text": "The secure API Gateway operates via a dedicated isolated proxy channel for a flat rate of $7.00."},
    {"id": "doc_004", "text": "The baseline program token pool limits are capped at a shared allocation of $315.00."}
]

print("\n--- Phase 1: Ingesting & Vectorizing Knowledge Corpus ---")
upsert_batch = []
for item in qa_knowledge_corpus:
    print(f" -> Slicing & Encoding Vector ID: {item['id']}")
    dense_vector = embedding_model.encode(item['text']).tolist()
    
    # Structure the payload with Text Metadata for our Grounding layer
    upsert_batch.append((item['id'], dense_vector, {"raw_text": item['text']}))

# Batch stream data over our API Gateway
index.upsert(vectors=upsert_batch)
print(f"Success: {len(qa_knowledge_corpus)} records committed to the index registry.")
time.sleep(2) # Allow cloud index synchronization buffer


--- Phase 1: Ingesting & Vectorizing Knowledge Corpus ---
 -> Slicing & Encoding Vector ID: doc_001
 -> Slicing & Encoding Vector ID: doc_002
 -> Slicing & Encoding Vector ID: doc_003
 -> Slicing & Encoding Vector ID: doc_004
Success: 4 records committed to the index registry.


In [4]:
# STEP 4: RETRIEVAL PHASE (SIMILARITY SEARCH)
print("\n--- Phase 2: Simulating Production User Query ---")
user_query = "What instance size runs our cluster and what is its cost?"
expected_ground_truth = "$11.65" # Our expected verification token

print(f"Incoming Query:     '{user_query}'")
print(f"Expected Answer:    '{expected_ground_truth}'")

# Vectorize the active inquiry
query_vector = embedding_model.encode(user_query).tolist()

# Requesting Top 3 results to test both precision and noise limits
search_output = index.query(vector=query_vector, top_k=3, include_metadata=True)


--- Phase 2: Simulating Production User Query ---
Incoming Query:     'What instance size runs our cluster and what is its cost?'
Expected Answer:    '$11.65'


In [5]:
# STEP 5: AI QUALITY ENGINEERING AUDIT LAYER
print("\n--- Phase 3: Executing Automated Quality Assurance Audit ---")
retrieved_records = search_output['matches']

hit_counter = 0
total_payload_chars = 0
meaningful_payload_chars = 0

print("\n[RANKING ANALYSIS LOOPS]:")
for position, record in enumerate(retrieved_records, start=1):
    chunk_text = record['metadata']['raw_text']
    chunk_length = len(chunk_text)
    total_payload_chars += chunk_length
    
    # Audit verification check
    has_answer = expected_ground_truth in chunk_text
    print(f" -> Position [{position}] | Distance Score: {record['score']:.4f} | Contains Answer: {has_answer}")
    
    if has_answer:
        hit_counter += 1
        meaningful_payload_chars += chunk_length


--- Phase 3: Executing Automated Quality Assurance Audit ---

[RANKING ANALYSIS LOOPS]:
 -> Position [1] | Distance Score: 0.4703 | Contains Answer: False
 -> Position [2] | Distance Score: 0.6943 | Contains Answer: True
 -> Position [3] | Distance Score: 0.4172 | Contains Answer: False


In [6]:
# STEP 6: CONSTRUCT THE FINAL PROGRAM EXECUTIVE COMPLIANCE REPORT
print("\n=========================================================")
print("     CAPSTONE AUDIT QUALITY ENGINEERING REPORT         ")
print("=========================================================")

# Metric 1: System Hit Rate
hit_rate_percentage = 100.0 if hit_counter > 0 else 0.0
print(f"SYSTEM RETRIEVAL HIT RATE:       {hit_rate_percentage:.1f}%")
if hit_rate_percentage == 100.0:
    print(" -> TARGET STATUS:               [PASS] (Answer surfaced in context window)")
else:
    print(" -> TARGET STATUS:               [FAIL] (Severe retrieval extraction breach)")

# Metric 2: System Noise Ratio
if total_payload_chars > 0:
    noise_ratio_percentage = ((total_payload_chars - meaningful_payload_chars) / total_payload_chars) * 100
else:
    noise_ratio_percentage = 100.0
print(f"TOTAL SYSTEM CONTEXT NOISE:      {noise_ratio_percentage:.2f}%")

# Architectural recommendation logic based on noise limits
if noise_ratio_percentage > 50.0:
    print(" -> RECOMMENDATION:              [ALERT] High noise detected. Deploy a Re-Ranker.")
else:
    print(" -> RECOMMENDATION:              [STABLE] Context window token usage optimized.")
print("=========================================================")


     CAPSTONE AUDIT QUALITY ENGINEERING REPORT         
SYSTEM RETRIEVAL HIT RATE:       100.0%
 -> TARGET STATUS:               [PASS] (Answer surfaced in context window)
TOTAL SYSTEM CONTEXT NOISE:      61.07%
 -> RECOMMENDATION:              [ALERT] High noise detected. Deploy a Re-Ranker.
